# Demo Trading Message Data and Limit Order Book

## Understand the Data

The description of the two data files can be found in *MessageandOrderREADME.txt*. Both files have no headers. 

These files provide the trading messages and limit order book data for the stock of Amazon.com, Inc. (Ticker AMZN), during the trading hours (9:30AM to 4PM, US Eastern Time) on one trading day, June 21, 2016.

In [1]:
import pandas as pd
df_orderbook = pd.read_csv('classdata/orderbook.csv',header=None)
df_message = pd.read_csv('classdata/message.csv',header=None)
df_message.columns=["Time","Type","OrderID","Size","Price","Direction"]

In [2]:
df_orderbook.shape

(1161, 40)

In [3]:
df_orderbook.head()

,0,1,2,3,4,5,6,7,8,9,...,30,31,32,33,34,35,36,37,38,39
0,2500100,0,2500000,150,2500200,0,2499900,0,2500300,0,...,2499300,0,2500900,0,2499200,0,2501000,0,2499100,0
1,2499900,0,2499700,0,2500000,0,2499600,0,2500100,0,...,2499000,0,2500700,0,2498900,0,2500800,0,2498800,0
2,2499900,0,2499500,50,2500000,0,2499400,0,2500100,0,...,2498800,0,2500700,0,2498700,0,2500800,0,2498600,0
3,2500100,0,2499900,0,2500200,0,2499800,0,2500300,0,...,2499200,0,2500900,0,2499100,0,2501000,0,2499000,0
4,2500100,0,2499500,10,2500200,0,2499400,0,2500300,0,...,2498800,0,2500900,0,2498700,0,2501000,0,2498600,0


The columns of *orderbook.csv*:

| Column | Meaning | Also called |
| --- | :--- | :--- |
| 0 | Level 1 Ask Price | Best Ask |
| 1 | Level 1 Ask Volume | Best Ask Volume |
| 2 | Level 1 Bid Price | Best Bid |
| 3 | Level 1 Bid Volume | Best Bid Volume |
| 4 | Level 2 Ask Price | 2nd Best Ask |
| 5 | Level 2 Ask Volume | 2nd Best Ask Volume |
| 6 | Level 2 Bid Price | 2nd Best Bid |
| 7 | Level 2 Bid Volume | 2nd Best Bid Volume |
| ... | ... | ... |
| ... | ... | ... |
| 36 | Level 10 Ask Price | 10th Best Ask |
| 37 | Level 10 Ask Volume | 10th Best Ask Volume |
| 38 | Level 10 Bid Price | 10th Best Bid |
| 39 | Level 10 Bid Volume | 10th Best Bid Volume |



In [4]:
df_message.shape

(1161, 6)

In [5]:
df_message.head()

,Time,Type,OrderID,Size,Price,Direction
0,34229.862418,1,16000000,150,2500000,1
1,34266.984822,3,16000000,150,2500000,1
2,34297.283137,1,16000010,50,2499500,1
3,34322.462282,3,16000010,50,2499500,1
4,34340.425193,1,16000019,10,2499500,1


The columns of *message.csv*:

    1.) Time: Number of seconds from 00:00am. For example, 34200 means 9:30am
    2.) Type: See below 
    3.) OrderID: A unique ID for each order (not for each message)
    4.) Size: Number of shares
    5.) Price: Dollar price times 10000
    6.) Direction: -1 means Sell; 1 means Buy 

The type of message:

                1: Submission of a new limit order
				2: Cancellation (Partial deletion of a limit order)
				3: Deletion (Total deletion of a limit order)
				4: Execution of a visible limit order			   	 
				5: Execution of a hidden limit order

Row **i** in **df_message** corresponds to row **i** of **df_orderbook**.

## Summary Statistics using Boolean Condition

In [6]:
#The frequency of each type of messages
print(df_message[df_message.Type==1].shape[0])
print(df_message[df_message.Type==2].shape[0])
print(df_message[df_message.Type==3].shape[0])
print(df_message[df_message.Type==4].shape[0])
print(df_message[df_message.Type==5].shape[0])

638
180
168
133
42


In [7]:
#The average price of each type of messages
print(df_message[df_message.Type==1].Price.mean())
print(df_message[df_message.Type==2].Price.mean())
print(df_message[df_message.Type==3].Price.mean())
print(df_message[df_message.Type==4].Price.mean())
print(df_message[df_message.Type==5].Price.mean())

2495130.250783699
2497202.777777778
2496732.1428571427
2497096.2406015038
2495821.4285714286


In [8]:
#The number of unique orders involved in each type of message
print(df_message[df_message.Type==1].OrderID.nunique())
print(df_message[df_message.Type==2].OrderID.nunique())
print(df_message[df_message.Type==3].OrderID.nunique())
print(df_message[df_message.Type==4].OrderID.nunique())
print(df_message[df_message.Type==5].OrderID.nunique())

638
138
168
133
1


# Pivot Table

How can we  identify all limited orders that are ultimately completely executed or completely cancelled. 

**An order is completely executed when the submitted size = the total executed size.**

**An order is completely cancelled when the submitted size = the total deleted size.**

To check these conditions for each order, we have to first create a two-way pivot table to facilitate the analysis.

In [9]:
import numpy as np
table=pd.pivot_table(df_message,
                     index=['OrderID'],  #Row variable of pivot table
                     columns=['Type'],   #Column variable of pivot table
                     values='Size',      #Data to be summarized in pivot table
                     fill_value=0,       #How to replace NaN
                     aggfunc=sum         #How to summarize the data (sum, len, or np.mean) 
                    )   
table[10020:10030]

/tmp/ipykernel_3774/159808386.py:2: FutureWarning: The provided callable <built-in function sum> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  table=pd.pivot_table(df_message,


Type,1,2,3,4,5
OrderID,,,,,


The IDs of the orders that were completely deleted:

In [10]:
ID_deleted = list(table[(table[1]==table[3]) & (table[1]>0)].index)
ID_deleted[0:5]

[16000000, 16000010, 16000019, 16000062, 16000100]

The IDs of the orders that were completely executed:

In [11]:
ID_executed = list(table[(table[1]==table[4]) & (table[1]>0)].index)
ID_deleted[0:5]

[16000000, 16000010, 16000019, 16000062, 16000100]

## Life time of an order

We can also use **groupby** to calculate the life time of each order.

In [12]:
#Order (OrderID 0 removed) and their lifetimes
df_temp=df_message[df_message.OrderID!=0]
df_temp.groupby(by="OrderID").Time.max()-df_temp.groupby(by="OrderID").Time.min()

OrderID
16000000     37.122404
16000010     25.179145
16000019     24.739918
16000062    109.757210
16000100     63.745711
               ...    
16015369      0.000000
16015388      0.000000
16015420      0.000000
16015462      0.000000
16015464      0.000000
Name: Time, Length: 638, dtype: float64

In [13]:
#Order (OrderID 0 removed) and their average lifetime
df_temp=df_message[df_message.OrderID!=0]
np.mean(df_temp.groupby(by="OrderID").Time.max()-df_temp.groupby(by="OrderID").Time.min())

2543.6425481018805

For example, if we want to see the average life time of all completely executed orders, we can do

In [14]:
df_temp = df_message[df_message.OrderID.isin(ID_executed)]
np.mean(df_temp.groupby(by="OrderID").Time.max()-df_temp.groupby(by="OrderID").Time.min())

3176.5455850882354

Another example, if we want to see the average life time of all deleted orders during 9:30AM to 10:30AM, we can do

In [15]:
df_temp = df_message[(df_message.Time>=34200) & (df_message.Time<=37800) & df_message.OrderID.isin(ID_deleted)]
np.mean(df_temp.groupby(by="OrderID").Time.max()-df_temp.groupby(by="OrderID").Time.min())

239.27495269999952

## Create bins for each minute

We can split messages and order books into bins (intervals) with each in a length of 1 minute.

In [16]:
import numpy as np
#np.arange(start,stop,stepsize) generate a sequence of spaced numbers
bins = np.arange(34200,57660,60)
df_orderbook['binned'] = pd.cut(df_message.Time, bins)
df_orderbook.head()

,0,1,2,3,4,5,6,7,8,9,...,31,32,33,34,35,36,37,38,39,binned
0,2500100,0,2500000,150,2500200,0,2499900,0,2500300,0,...,0,2500900,0,2499200,0,2501000,0,2499100,0,"(34200, 34260]"
1,2499900,0,2499700,0,2500000,0,2499600,0,2500100,0,...,0,2500700,0,2498900,0,2500800,0,2498800,0,"(34260, 34320]"
2,2499900,0,2499500,50,2500000,0,2499400,0,2500100,0,...,0,2500700,0,2498700,0,2500800,0,2498600,0,"(34260, 34320]"
3,2500100,0,2499900,0,2500200,0,2499800,0,2500300,0,...,0,2500900,0,2499100,0,2501000,0,2499000,0,"(34320, 34380]"
4,2500100,0,2499500,10,2500200,0,2499400,0,2500300,0,...,0,2500900,0,2498700,0,2501000,0,2498600,0,"(34320, 34380]"


In [17]:
#The last order book in each minute
df_orderbook_minute = df_orderbook.groupby(by="binned").tail(1).copy()
df_orderbook_minute.head()

/tmp/ipykernel_3774/3158550412.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_orderbook_minute = df_orderbook.groupby(by="binned").tail(1).copy()


,0,1,2,3,4,5,6,7,8,9,...,31,32,33,34,35,36,37,38,39,binned
0,2500100,0,2500000,150,2500200,0,2499900,0,2500300,0,...,0,2500900,0,2499200,0,2501000,0,2499100,0,"(34200, 34260]"
2,2499900,0,2499500,50,2500000,0,2499400,0,2500100,0,...,0,2500700,0,2498700,0,2500800,0,2498600,0,"(34260, 34320]"
6,2500400,200,2499800,0,2500500,0,2499700,0,2500600,0,...,0,2501200,0,2499000,0,2501300,0,2498900,0,"(34320, 34380]"
9,2499900,200,2499700,0,2500100,50,2499600,0,2500400,250,...,0,2501000,0,2498900,0,2501100,0,2498800,0,"(34380, 34440]"
11,2500100,50,2499700,0,2500400,50,2499600,0,2500500,0,...,0,2501100,0,2498900,0,2501200,0,2498800,0,"(34440, 34500]"


Can we use **np.arange** to generate the column indexes for the bid sizes and ask sizes in df_orderbook?

If yes, you will be able to use those column IDs to extract the columns of bid sizes and ask sizes separately. Then you can use .sum(axis=1) to calculate the sum of each row, which is the total number of buy or sell shares. 

For example:

In [18]:
#Create column indexes
import numpy as np
askvolumeid = np.arange(1,40,4)
bidvolumeid = np.arange(3,40,4)
print(askvolumeid)
print(bidvolumeid)

[ 1  5  9 13 17 21 25 29 33 37]
[ 3  7 11 15 19 23 27 31 35 39]


In [19]:
#Get the total bid size and the toal ask size
df_orderbook_minute["totalbid"]=df_orderbook_minute.iloc[:,bidvolumeid].sum(axis=1)
df_orderbook_minute["totalask"]=df_orderbook_minute.iloc[:,askvolumeid].sum(axis=1)
df_orderbook_minute.head()

,0,1,2,3,4,5,6,7,8,9,...,33,34,35,36,37,38,39,binned,totalbid,totalask
0,2500100,0,2500000,150,2500200,0,2499900,0,2500300,0,...,0,2499200,0,2501000,0,2499100,0,"(34200, 34260]",150,0
2,2499900,0,2499500,50,2500000,0,2499400,0,2500100,0,...,0,2498700,0,2500800,0,2498600,0,"(34260, 34320]",50,0
6,2500400,200,2499800,0,2500500,0,2499700,0,2500600,0,...,0,2499000,0,2501300,0,2498900,0,"(34320, 34380]",0,200
9,2499900,200,2499700,0,2500100,50,2499600,0,2500400,250,...,0,2498900,0,2501100,0,2498800,0,"(34380, 34440]",0,500
11,2500100,50,2499700,0,2500400,50,2499600,0,2500500,0,...,0,2498900,0,2501200,0,2498800,0,"(34440, 34500]",0,100


In [20]:
#Midquote
df_orderbook_minute['Midquote'] = (df_orderbook_minute.iloc[:,0]+df_orderbook_minute.iloc[:,2])/2
df_orderbook_minute.head()

,0,1,2,3,4,5,6,7,8,9,...,34,35,36,37,38,39,binned,totalbid,totalask,Midquote
0,2500100,0,2500000,150,2500200,0,2499900,0,2500300,0,...,2499200,0,2501000,0,2499100,0,"(34200, 34260]",150,0,2500050.0
2,2499900,0,2499500,50,2500000,0,2499400,0,2500100,0,...,2498700,0,2500800,0,2498600,0,"(34260, 34320]",50,0,2499700.0
6,2500400,200,2499800,0,2500500,0,2499700,0,2500600,0,...,2499000,0,2501300,0,2498900,0,"(34320, 34380]",0,200,2500100.0
9,2499900,200,2499700,0,2500100,50,2499600,0,2500400,250,...,2498900,0,2501100,0,2498800,0,"(34380, 34440]",0,500,2499800.0
11,2500100,50,2499700,0,2500400,50,2499600,0,2500500,0,...,2498900,0,2501200,0,2498800,0,"(34440, 34500]",0,100,2499900.0


## Correlation

In [21]:
#How to compute the correlation between two sequences of numbers
data1=[3,1,2,3,2,4,1]
data2=[-2,-4,-4,-10,-5,-3,-7]
np.corrcoef(data1, data2)

array([[1.        , 0.16593663],
       [0.16593663, 1.        ]])